In [4]:
import numpy as np
import torch
import torch.nn.functional as F

def max_pool2d_forward(x: np.ndarray, kernel_size: int, stride: int = 1, padding: int = 0):
    N, C, H, W = x.shape
    # 关键修改：padding填充负无穷，不是0，和pytorch maxpool规则一致
    x_pad = np.pad(x, pad_width=((0, 0), (0, 0), (padding, padding), (padding, padding)), 
                   mode="constant", constant_values=-np.inf)
    out_h = (H + 2 * padding - kernel_size) // stride + 1
    out_w = (W + 2 * padding - kernel_size) // stride + 1
    out = np.zeros((N, C, out_h, out_w), dtype=x.dtype)

    for n in range(N):
        for c in range(C):
            for oh in range(out_h):
                h_start = oh * stride
                h_end = h_start + kernel_size
                for ow in range(out_w):
                    w_start = ow * stride
                    w_end = w_start + kernel_size
                    window = x_pad[n, c, h_start:h_end, w_start:w_end]
                    out[n, c, oh, ow] = np.max(window)
    return out

if __name__ == "__main__":
    np.random.seed(123)
    input_np = np.random.randn(2, 3, 6, 6).astype(np.float32)
    k_size = 2
    s = 2
    p = 1

    my_pool_out = max_pool2d_forward(input_np, kernel_size=k_size, stride=s, padding=p)

    input_torch = torch.from_numpy(input_np)
    torch_out = F.max_pool2d(input_torch, kernel_size=k_size, stride=s, padding=p)

    print("====自定义MaxPool输出形状====")
    print(my_pool_out.shape)
    print("\n====Pytorch官方MaxPool输出形状====")
    print(torch_out.shape)
    print("\n====前2行2列数值对比【自定义】====")
    print(my_pool_out[0, 0, :2, :2])
    print("\n====前2行2列数值对比【官方torch】====")
    print(torch_out[0, 0, :2, :2].numpy())
    print("\n====结果是否完全相等====")
    print(np.allclose(my_pool_out, torch_out.numpy()))

====自定义MaxPool输出形状====
(2, 3, 4, 4)

====Pytorch官方MaxPool输出形状====
torch.Size([2, 3, 4, 4])

====前2行2列数值对比【自定义】====
[[-1.0856307   0.99734545]
 [ 1.4913896   1.2659363 ]]

====前2行2列数值对比【官方torch】====
[[-1.0856307   0.99734545]
 [ 1.4913896   1.2659363 ]]

====结果是否完全相等====
True


In [7]:
import torch
import torch.nn as nn

# 题目要求：定义NiN块，参数in_channels/out_channels/kernel_size/stride/padding，Sequential、1普通Conv+2个1×1Conv、每层后ReLU
def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    block = nn.Sequential(
        # 第一个自定义参数卷积
        nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding),
        nn.ReLU(),
        # 第一个1×1卷积
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        # 第二个1×1卷积
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU()
    )
    return block

# 自测代码（运行自动输出shape，提交作业可删掉下面if里内容）
if __name__ == "__main__":
    # 构造测试输入：batch=2，通道3，H=32,W=32
    test_input = torch.randn(2, 3, 32, 32)
    # 实例化NiN块
    block = nin_block(in_channels=3, out_channels=64, kernel_size=3, stride=1, padding=1)
    out = block(test_input)
    print("NiN Block输出形状：", out.shape)

NiN Block输出形状： torch.Size([2, 64, 32, 32])


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 自定义残差块Residual类
class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, use_1x1conv=False):
        super(Residual, self).__init__()
        # 第一个3×3卷积+BN
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, stride=stride, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        # 第二个3×3卷积+BN，输出通道和上一层一致
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, stride=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # use_1x1conv为True时构建1×1卷积调整输入通道与尺寸
        self.conv_shortcut = None
        if use_1x1conv:
            self.conv_shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False)

    def forward(self, x):
        # 主分支f(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        # 旁路调整输入x
        if self.conv_shortcut is not None:
            x = self.conv_shortcut(x)

        # 残差相加 f(x)+x
        out = out + x
        out = F.relu(out)
        return out

# 测试代码
if __name__ == '__main__':
    # 测试1：通道不一致，开启1×1卷积
    blk1 = Residual(in_channels=3, out_channels=64, stride=1, use_1x1conv=True)
    x1 = torch.randn(2, 3, 32, 32)
    y1 = blk1(x1)
    print("开启1×1卷积输出形状：", y1.shape)

    # 测试2：通道一致，不使用1×1卷积
    blk2 = Residual(in_channels=64, out_channels=64, stride=1, use_1x1conv=False)
    x2 = torch.randn(2, 64, 32, 32)
    y2 = blk2(x2)
    print("不使用1×1卷积输出形状：", y2.shape)

开启1×1卷积输出形状： torch.Size([2, 64, 32, 32])
不使用1×1卷积输出形状： torch.Size([2, 64, 32, 32])


In [10]:
import torch
from torchvision import transforms

# 组合图像增广管道（Pipeline）
def get_image_augmentation_pipeline():
    # 组合所有变换
    transform = transforms.Compose([
        # 1. 随机裁剪面积 0.08~1.0，缩放为 224x224
        transforms.RandomResizedCrop(
            size=224,
            scale=(0.08, 1.0)
        ),
        
        # 2. 50% 概率水平翻转
        transforms.RandomHorizontalFlip(p=0.5),
        
        # 3. 随机改变亮度、对比度、饱和度，范围 0.5
        transforms.ColorJitter(
            brightness=0.5,
            contrast=0.5,
            saturation=0.5
        ),
        
        # 4. 转换为 PyTorch 张量
        transforms.ToTensor()
    ])
    return transform

# ------------------- 测试代码（运行即可看到结果）-------------------
if __name__ == "__main__":
    # 获取增广管道
    aug_pipeline = get_image_augmentation_pipeline()
    print("图像增广管道创建成功！")
    print("包含操作：")
    print("1. 随机裁剪+缩放至224×224")
    print("2. 50%概率水平翻转")
    print("3. 随机亮度、对比度、饱和度（0.5）")
    print("4. 转为Tensor")

图像增广管道创建成功！
包含操作：
1. 随机裁剪+缩放至224×224
2. 50%概率水平翻转
3. 随机亮度、对比度、饱和度（0.5）
4. 转为Tensor


In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def label_smoothing_cross_entropy(preds, targets, epsilon=0.1):
    """
    标签平滑交叉熵损失函数（完全符合题目定义）
    :param preds: 模型输出 logits (batch_size, num_classes)
    :param targets: 真实标签 (batch_size,)  —— 类别索引
    :param epsilon: 标签平滑因子，默认 0.1
    :return: 标量损失值
    """
    # 1. 获取分类数量 K
    num_classes = preds.size(1)
    
    # 2. 对模型输出做 log_softmax 归一化
    log_preds = F.log_softmax(preds, dim=1)

    # 3. 构建标签平滑后的真实分布
    # 真实类概率：1 - ε
    # 其他类概率：ε / (K - 1)
    true_dist = torch.zeros_like(log_preds)
    true_dist.fill_(epsilon / (num_classes - 1))  # 先填充所有类为 ε/(K-1)
    true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - epsilon)  # 真实类设为 1-ε

    # 4. 计算交叉熵损失（均值）
    loss = (-true_dist * log_preds).sum(dim=1).mean()
    
    return loss


# ------------------- 测试代码（运行即可验证）-------------------
if __name__ == "__main__":
    # 模拟：batch=2，10分类任务
    batch_size = 2
    num_classes = 10

    # 模型输出 logits
    preds = torch.randn(batch_size, num_classes)
    # 真实标签
    targets = torch.randint(0, num_classes, (batch_size,))

    # 计算标签平滑损失
    loss = label_smoothing_cross_entropy(preds, targets, epsilon=0.1)

    print("模型预测形状：", preds.shape)
    print("真实标签：", targets)
    print("标签平滑交叉熵损失：", loss.item())

模型预测形状： torch.Size([2, 10])
真实标签： tensor([0, 8])
标签平滑交叉熵损失： 1.613107442855835
